# SDV SYNTHESIZERS SYNTHETIC DATA EVALUATION

## LOAD SYNTHESIZERS & ORIGINAL FILE

### Unrar synthesizers 

In [ ]:
import patoolib
import os
import pandas as pd
import joblib

# set path
results_folder = "../results"

# unzip synthesizers .rar file
patoolib.extract_archive(os.path.join(results_folder,"results_synthesizers.rar"), outdir=results_folder)

### Load GaussianCopula synthesizers

In [ ]:
# GAUSSIAN COPULA SYNTHESIZERS
gcopula_kde = joblib.load(os.path.join(results_folder,"synth_kde.joblib"))
gcopula_rest = joblib.load(os.path.join(results_folder,"synth_rest.joblib"))

# add to list both gaussian copula synthesizers
gcopula_synths = [gcopula_rest, gcopula_kde]
print(gcopula_synths)

### Load CTGAN synthesizer

In [ ]:
# GAUSSIAN COPULA SYNTHESIZERS
ctgan_synthesizer = joblib.load(os.path.join(results_folder,"ctgan_synthesizer.joblib"))

### Load original file

In [ ]:
# load file
diabetes = pd.read_parquet(os.path.join(results_folder,"preprocessed_file.parquet"), engine = "pyarrow")

# visualize results
diabetes.head()

## GET METADATA

In [ ]:
from sdv.metadata import SingleTableMetadata

# def function
def create_metadata(df):
    """
    SingleTableMetadata type data creation. Obtains information directly from original dataframe.
    
    Parameters:
        df (pd.DataFrame): The original DataFrame.

    Returns:
        SingleTableMetadata: metadata to create synthetic data
    """
    # Automatically detect metadata from the actual DataFrame
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df)

    return metadata

# call to function
metadata = create_metadata(diabetes)

# visualize result
metadata

## CREATE N SYNTHETIC DATA & SELECT BEST

In [17]:
from sdv.evaluation.single_table import evaluate_quality

# define functions
def generate_synthetic_data_by_2_synths(synth_full, synth_kde, num_rows):
    """
    Generates synthetic data from both synthesizers.

    Parameters:
        synth_full: synthesizer for categorical and gamma distributed columns
        synth_kde: synthesizer for gaussian_kde columns
        num_rows: Number of rows for synthetic data

    Returns:
        pd.DataFrame: Combined synthetic data.
    """
    # Generate synthetic data
    synthetic_full = synth_full.sample(num_rows)    
    synthetic_kde = synth_kde.sample(num_rows)

    # Combine the dataframes, aligning by columns
    synthetic_data = pd.concat([synthetic_full.reset_index(drop=True), synthetic_kde.reset_index(drop=True)], axis=1)

    return synthetic_data

def create_synth_data (df, synth):
    """
    Generates synthetic data based on passed synthesizer.

    Parameters:
        synth: synthesizer 

    Returns:
        pd.DataFrame: Combined synthetic data.
    """

    # create new synth data
    synthetic_data = synth.sample(
        num_rows=df.shape[0]
    )
    
    return synthetic_data


def quality_evaluation(quality, synth_data, best_synth_data, overall_qa, col_shape_qa, col_pairs_qa, verbose = False):  
    """
    Function that evaluates actual quality data with previous one and set as best only if all values are performed.

    Parameters:
        quality: sdv quality-report.
        synth_data (dataframe): actual synth data to evaluate
        best_synth_data (dataframe): best synth data until now
        overall_qa (float): overall score.
        col_shape_qa (float): columns shape score.
        col_pairs_qa (float): columns pairs score.
        verbose (boolean): set if log need to be plot or not. Default value: False.

    Returns:
        best_synth_data (dataframe): best synth data until now
        overall_qa (float): best synth data's score.
        col_shape_qa (float): best synth data's columns shape score.
        col_pairs_qa (float): best synth data's columns pairs score.
    """

    # get the 3 indicators and compare
    aux_overall = quality.get_score()
    aux_shape = quality.get_properties().iloc[0]["Score"]
    aux_pair = quality.get_properties().iloc[1]["Score"]

    # compare actual with previous
    if (aux_overall >= overall_qa) and (aux_shape >= col_shape_qa) and (aux_pair >= col_pairs_qa):
        overall_qa = aux_overall
        col_shape_qa = aux_shape
        col_pairs_qa = aux_pair
        best_synth_data = synth_data
        
        if verbose:
            print(f"\nOverall quality: {aux_overall} >= {overall_qa}")
            print(f"Columns shape quality: {aux_shape} >= {col_shape_qa}")
            print(f"Columns pair quality: {aux_pair} >= {col_pairs_qa}")
            print(f"Best synth_data: {best_synth_data}")
        
    return overall_qa, col_shape_qa, col_pairs_qa, best_synth_data

def evaluate_best_synth_data(df, md, synth_type, synthesizers, num_synth_data, verbose = False):
    """
    Function that creates and evaluates N created synthetic data and returns the best.

    Parameters:
        df (datafame): original dataframe to compare
        md: metadata
        synth_type (str): String that represent what kind of synthesizer need to be used to create synthetic data.
        GaussianCopula & CTGAN are only possible.
        best_synth_data (dataframe): best synth data until now
        synthesizers (list): List of synthesizers to take into account to create the synthetic data.
        num_synth_data (int): Number of synthetic data to create and evaluate.
        verbose (boolean): set if log need to be plot or not. Default value: False.

    Returns:
        best_synth_data (dataframe): best synth data.
        overall_qa (float): best synth data's score.
        col_shape_qa (float): best synth data's columns shape score.
        col_pairs_qa (float): best synth data's columns pairs score.
    """

    overall_qa = 0.0
    col_shape_qa = 0.0
    col_pairs_qa = 0.0
    best_synth_data = None
        
    # create num_synth_data n synth data
    for i in range(0, num_synth_data):
        
        print(f"\nSynthetic data creation #{(i+1)} for {synth_type}")
        
        if synth_type == "GaussianCopula" and len(synthesizers) == 2:
            # generate synthetic data 
            synth_data = generate_synthetic_data_by_2_synths(synthesizers[0], 
                                                             synthesizers[1],
                                                             df.shape[0]) # GaussianCopula                    
            print(f"Synthetic data created. Evaluating...")
            
            # evaluate     
            quality = evaluate_quality(df, synth_data, md)  
            overall_qa, col_shape_qa, col_pairs_qa, best_synth_data = quality_evaluation(quality, synth_data, 
                                                                                         best_synth_data,
                                                                                         overall_qa, 
                                                                                         col_shape_qa, 
                                                                                         col_pairs_qa,
                                                                                         verbose)
        elif synth_type == "CTGAN" and len(synthesizers) == 1:
            
            # generate synth data
            synth_data = create_synth_data(df, synthesizers[0]) # CTGAN
            print(f"Synthetic data created. Evaluating...")
            
            # evaluate     
            quality = evaluate_quality(df, synth_data, md)  
            overall_qa, col_shape_qa, col_pairs_qa, best_synth_data = quality_evaluation(quality, synth_data,                                                                                         
                                                                                         best_synth_data,
                                                                                         overall_qa, 
                                                                                         col_shape_qa, 
                                                                                         col_pairs_qa,
                                                                                         verbose)

    return overall_qa, col_shape_qa, col_pairs_qa, best_synth_data

### With GaussianCopulaSynthesizer

In [ ]:
# Create 3 GaussianCopula synthetic data and obtain best
overall_qa, col_shape_qa, col_pairs_qa, best_synth_data = evaluate_best_synth_data(df = diabetes, md = metadata, 
                                                                                   synth_type = "GaussianCopula", 
                                                                                   synthesizers = gcopula_synths, 
                                                                                   num_synth_data = 3)

# visualize result
print(f"Best synth_data: {best_synth_data.head()}")
print(f"Overall quality: {overall_qa}")
print(f"Columns shape quality: {col_shape_qa}")
print(f"Columns pair quality: {col_pairs_qa}")

### Save best GaussianCopula synth data in file

In [ ]:
# save best data as file
best_synth_data.to_parquet(os.path.join(results_folder, "GaussianCopula_best_synth_data.parquet"), engine = "pyarrow")

### With CTGANSynthesizer

In [ ]:
# Create 3 CTGAN synthetic data and obtain best
overall_qa, col_shape_qa, col_pairs_qa, best_synth_data = evaluate_best_synth_data(df = diabetes, md = metadata, 
                                                                                   synth_type = "CTGAN", 
                                                                                   synthesizers = [ctgan_synthesizer], 
                                                                                   num_synth_data = 3)

# visualize result
print(f"Best synth_data: {best_synth_data.head()}")
print(f"Overall quality: {overall_qa}")
print(f"Columns shape quality: {col_shape_qa}")
print(f"Columns pair quality: {col_pairs_qa}")

### Save best CTGAN synth data in file

In [ ]:
# save best data as file
best_synth_data.to_parquet(os.path.join(results_folder, "CTGAN_best_synth_data.parquet"), engine = "pyarrow")

## SAVE ALL BEST SYNTH DATA IN RAR FILE

In [ ]:
import os
import subprocess

def save_file_as_rar (output_path, file_paths):
     #Prepare the rar command
    command = ['rar', 'a', output_path] + file_paths

    # Execute the command
    subprocess.run(command, check=True)


# Set paths to synthetizers
file1 = os.path.join(results_folder,"GaussianCopula_best_synth_data.parquet")
file2 = os.path.join(results_folder,"CTGAN_best_synth_data.parquet")

# join all files to add to RAR file in a list
file_paths = [file1, file2]  

# set where to save RAR file and name
output_path = os.path.join(results_folder,"best_synth_data.rar")  

# call to function
save_file_as_rar (output_path, file_paths)